# EEG features: target-locked P3a and cue-locked CNV

This notebook extracts single-trial EEG amplitudes, prepares participant summaries, and matches P3a and CNV responses from the same trials. It also saves condition averages for subsequent waveform and scalp-map figures.

## Inputs

- **P3a recordings:** continuous FIF files prepared for the target-locked analysis (1–30 Hz).
- **CNV recordings:** separate continuous FIF files prepared for slow-potential analysis (0.05–30 Hz).
- **Reviewed saccade summary:** `saccades_summary.xlsx` produced by `01_saccades.ipynb`, including its `Trial_annotations` sheet.

The FIF recordings must already have the intended EEG reference and artifact-component correction applied. This notebook performs epoching, baseline correction, and epoch rejection; the input signals are not filtered or resampled here. The sampling frequency is read from each file. A CNV recording requires preservation of slow activity during preprocessing.

Both EEG versions must retain the complete target-event sequence in the same sample coordinate system. Reviewed saccades are matched only after checking the full target count, event order, and trial identifiers. P3a and CNV trials are joined by participant and target sample after verifying both EEG time bases and target sequences.

## Run instructions

1. Set both FIF directories, their filename patterns, and `SACCADE_SUMMARY_FILE`.
2. Check the cohort identifiers and the CNV-specific participant exclusions.
3. Set `RUN_STUDY_ANALYSIS = True`, restart the kernel, and run all cells.
4. Inspect the input metadata, participant/trial counts, rejection reasons, paired-trial counts, and diagnostic waveforms.
5. Set `OUTPUT_DIR` and `EXPORT_OUTPUTS = True`, then run the export cell.

With study analysis disabled, the notebook loads no participant data. Python dependencies are `mne`, `numpy`, `pandas`, `matplotlib`, `openpyxl`, and Jupyter.

In [2]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import platform
import re

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

software_versions = pd.DataFrame(
    [("Python", platform.python_version())]
    + [(name, version(name)) for name in ("mne", "numpy", "pandas", "matplotlib", "openpyxl")],
    columns=["software", "version"],
)
pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 18)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
display(software_versions)

,software,version
0,Python,3.13.9
1,mne,1.11.0
2,numpy,2.3.5
3,pandas,2.3.3
4,matplotlib,3.10.6
5,openpyxl,3.1.5


In [3]:
RUN_STUDY_ANALYSIS = False
ERP_FIF_DIR = Path("PATH_TO_P3A_FIF").expanduser()
CNV_FIF_DIR = Path("PATH_TO_CNV_FIF").expanduser()
SACCADE_SUMMARY_FILE = Path("PATH_TO_SACCADE_OUTPUTS/saccades_summary.xlsx").expanduser()
OUTPUT_DIR = Path("PATH_TO_EEG_OUTPUTS").expanduser()

# Each pattern must identify exactly one continuous recording per participant.
ERP_FILE_PATTERN = "{subject}*ica.fif"
CNV_FILE_PATTERN = "{subject}*.fif"
EXPORT_OUTPUTS = False
SAVE_EVOKEDS = True
OVERWRITE_OUTPUTS = False
SHOW_QC_PLOTS = True
QC_PARTICIPANT = "R01"

EXPECTED_N_PARTICIPANTS = 69
PARTICIPANT_IDS = """
R01 R02 R03 R06 R08 R09 R10 R12 R13
R14 R15 R16 R17 R20 R21 R22 R24 R25
R26 R27 R28 R29 R30 R31 R32 R33 R34
R35 R37 R38 R39 R40 R41 S01 S07 S09
S10 S12 S13 S20 S22 S27 S28 S29 S30
S35 S38 S43 S44 S45 S47 S49 Z01 Z02
Z03 Z04 Z05 Z06 Z07 Z09 Z10 Z11 Z12
Z13 Z15 Z16 Z17 Z19 Z20
""".split()

ERP_TMIN, ERP_TMAX = -0.2, 0.7
CNV_TMIN, CNV_TMAX = -0.2, 1.2
BASELINE = (-0.2, 0.0)
EEG_REJECT = {"eeg": 100e-6}
ERP_ROI = ["F3", "Fz", "F4", "Cz", "C3", "C4"]
ERP_WINDOW = (0.240, 0.320)
CNV_WINDOW = (0.350, 1.000)
CNV_REQUIRED_CHANNELS = ["Fz", "F3", "F4", "C3", "C4", "Cz", "P3", "P4", "T5", "T6", "O1", "O2"]
SOA_RANGE = (1.25, 1.60)
MIN_CNV_TRIALS = 30
CONDITION_ORDER = ["valid_80", "invalid_80", "valid_50", "invalid_50"]

if EXPORT_OUTPUTS and not RUN_STUDY_ANALYSIS:
    raise ValueError("Enable RUN_STUDY_ANALYSIS before exporting study results.")
print("Mode:", "study analysis" if RUN_STUDY_ANALYSIS else "no study data loaded")

Mode: no study data loaded


## Measurement definitions

| Setting | P3a | CNV |
|---|---|---|
| Epoch reference | Target onset | Cue onset |
| Epoch interval | −200 to +700 ms | −200 to +1200 ms |
| Baseline | −200 to 0 ms | −200 to 0 ms |
| Amplitude | Mean across six ROI electrodes, 240–320 ms | Mean at Fz, 350–1000 ms |
| EEG rejection | Peak-to-peak amplitude exceeding 100 µV in an EEG channel considered by MNE; bad annotations rejected | Same |
| Saccade criterion | All EEG-retained target trials | Detected saccade with latency ≥0 ms relative to the target, regardless of direction correctness |
| Participant threshold | No additional minimum-trial threshold | At least 30 retained trials per probability context, pooling cue directions |

The amplitude windows use inclusive comparisons with the epoch time vector. The selected sample times are recorded. Amplitudes are exported in microvolts.

Marked bad channels, active projections, reference status, and filter metadata are recorded for inspection. Filter metadata alone do not document the complete preprocessing procedure. The three specified CNV participant exclusions do not apply to P3a.

In [4]:
EVENT_INFO = pd.DataFrame(
    [(20, 80, "valid", -1, -1), (21, 80, "invalid", 1, -1),
     (22, 50, "valid", -1, -1), (23, 50, "invalid", 1, -1),
     (30, 80, "valid", 1, 1), (31, 80, "invalid", -1, 1),
     (32, 50, "valid", 1, 1), (33, 50, "invalid", -1, 1)],
    columns=["event", "probability", "validity", "cue_direction", "target_direction"],
).set_index("event")
EVENT_INFO["condition"] = EVENT_INFO["validity"] + "_" + EVENT_INFO["probability"].astype(str)
CUE_DIRECTIONS = {12: -1, 13: 1}
TASK_PATTERN = re.compile(r"(?<!\d)(12|13|20|21|22|23|30|31|32|33)(?!\d)")
display(EVENT_INFO)

,probability,validity,cue_direction,target_direction,condition
event,,,,,
20,80,valid,-1,-1,valid_80
21,80,invalid,1,-1,invalid_80
22,50,valid,-1,-1,valid_50
23,50,invalid,1,-1,invalid_50
30,80,valid,1,1,valid_80
31,80,invalid,-1,1,invalid_80
32,50,valid,1,1,valid_50
33,50,invalid,-1,1,invalid_50


## Input validation and file metadata

The reviewed summary supplies the full trial sequence, including undetected and pretarget saccades. The CNV timing flag is calculated from the reviewed latency and direction. A latency of 0–89 ms passes this timing criterion.

Recordings are matched to the explicitly configured cohort. Missing or ambiguous input files stop processing. Missing required CNV channels are recorded as a CNV exclusion.

In [4]:
def load_saccade_summary(path, participant_ids):
    table = pd.read_excel(path, sheet_name="Trial_annotations", engine="openpyxl")
    required = {"subject", "trial", "event", "latency_ms", "direction"}
    if missing := required - set(table.columns):
        raise ValueError(f"Missing saccade columns: {sorted(missing)}")
    out = table[["subject", "trial", "event", "latency_ms", "direction"]].copy()
    out["subject"] = out["subject"].astype(str).str.strip().str.upper()
    if set(out["subject"]) != set(participant_ids):
        raise ValueError("The saccade summary and configured EEG cohort contain different participants.")
    for column in ("trial", "event", "latency_ms", "direction"):
        out[column] = pd.to_numeric(out[column], errors="raise")
    if not (out["trial"].notna() & np.isfinite(out["trial"]) & out["trial"].ge(0)
            & out["trial"].eq(np.floor(out["trial"]))).all():
        raise ValueError("Trial identifiers must be nonnegative integers.")
    out["trial"] = out["trial"].astype(int)
    if out.duplicated(["subject", "trial"]).any() or not out["event"].isin(EVENT_INFO.index).all():
        raise ValueError("Duplicate trial identifiers or unknown target event codes.")
    if np.isinf(out["latency_ms"]).any():
        raise ValueError("Saccade latencies must be finite or missing.")
    direction_ok = out["direction"].isin([-1, 1])
    if not (out["direction"].isna() | out["direction"].isin([-1, 0, 1])).all():
        raise ValueError("Saccade direction must be -1, 0, +1, or missing.")
    if (np.isfinite(out["latency_ms"]) != direction_ok).any():
        raise ValueError("A reviewed latency and direction are inconsistent.")
    out["saccade_detected"] = np.isfinite(out["latency_ms"]) & direction_ok
    out["cnv_saccade_ok"] = out["saccade_detected"] & out["latency_ms"].ge(0)
    return out


def resolve_recording(directory, pattern, subject):
    directory = Path(directory)
    if not directory.is_dir():
        raise FileNotFoundError(f"Recording directory does not exist: {directory}")
    matches = sorted(path for path in directory.glob(pattern.format(subject=subject)) if path.is_file())
    if len(matches) != 1:
        raise ValueError(f"{subject}: expected one recording for {pattern!r}, found {len(matches)}.")
    return matches[0]


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_prepared_recording(path):
    raw = mne.io.read_raw_fif(path, preload=True, verbose="ERROR")
    ear_channels = {ch: "misc" for ch in ("A1", "A2") if ch in raw.ch_names}
    if ear_channels:
        raw.set_channel_types(ear_channels, verbose="ERROR")
    return raw


def recording_metadata(raw, path, subject, analysis):
    return {
        "subject": subject, "analysis": analysis, "file": Path(path).name,
        "sha256": sha256_file(path), "sfreq_hz": float(raw.info["sfreq"]),
        "first_samp": int(raw.first_samp), "n_samples": int(raw.n_times),
        "highpass_hz": raw.info["highpass"], "lowpass_hz": raw.info["lowpass"],
        "custom_ref_applied": int(raw.info["custom_ref_applied"]),
        "bad_channels": "; ".join(raw.info["bads"]),
        "projector_count": len(raw.info["projs"]),
        "active_projector_count": sum(bool(p["active"]) for p in raw.info["projs"]),
    }

## Event sequence and trial alignment

Annotation descriptions are decoded to task event labels; MNE's internal integer event identifiers are kept separate. Target events are numbered from zero before EEG rejection.

The complete reviewed target sequence must match each recording in count and event order, with contiguous zero-based trial identifiers. A sequence mismatch stops processing. The P3a and CNV recordings must also have equal sampling frequencies, first-sample indices, and target samples. Inputs with independent cropping or resampling require alignment before this notebook is run.

In [5]:
def recording_events(raw):
    events, event_id = mne.events_from_annotations(raw, verbose="ERROR")
    inverse = {value: name for name, value in event_id.items()}
    rows = []
    for index, event in enumerate(events):
        description = inverse[int(event[2])]
        matches = TASK_PATTERN.findall(description)
        if len(matches) > 1:
            raise ValueError(f"Ambiguous task annotation: {description!r}")
        rows.append({"annotation_event_index": index, "sample": int(event[0]),
                     "event": int(matches[0]) if matches else np.nan,
                     "description": description})
    table = pd.DataFrame(rows, columns=["annotation_event_index", "sample", "event", "description"])
    return events, event_id, table


def align_targets(event_table, reviewed, subject):
    targets = event_table.loc[event_table["event"].isin(EVENT_INFO.index)].copy().reset_index(drop=True)
    if targets["sample"].duplicated().any():
        raise ValueError(f"{subject}: target annotations share a sample; inspect event alignment.")
    reviewed = reviewed.loc[reviewed["subject"].eq(subject)].copy()
    if len(targets) != len(reviewed):
        raise ValueError(f"{subject}: {len(targets)} EEG targets versus {len(reviewed)} reviewed trials.")
    if not np.array_equal(reviewed["trial"].to_numpy(), np.arange(len(targets))):
        raise ValueError(f"{subject}: reviewed trial order is not the complete zero-based sequence.")
    if not np.array_equal(targets["event"].to_numpy(), reviewed["event"].to_numpy()):
        raise ValueError(f"{subject}: EEG and reviewed target-event sequences differ.")
    targets = targets.rename(columns={"sample": "target_sample"}).drop(columns="description")
    targets.insert(0, "subject", subject)
    targets["trial"] = reviewed["trial"].to_numpy()
    targets["event"] = targets["event"].astype(int)
    for column in EVENT_INFO:
        targets[column] = targets["event"].map(EVENT_INFO[column])
    for column in ("latency_ms", "direction", "saccade_detected", "cnv_saccade_ok"):
        targets[column] = reviewed[column].to_numpy()
    return targets


def verify_eeg_time_base(erp_raw, cnv_raw, erp_targets, cnv_targets, subject):
    same_grid = (erp_raw.info["sfreq"] == cnv_raw.info["sfreq"]
                 and erp_raw.first_samp == cnv_raw.first_samp)
    same_targets = (np.array_equal(erp_targets["target_sample"], cnv_targets["target_sample"])
                    and np.array_equal(erp_targets["event"], cnv_targets["event"]))
    if not same_grid or not same_targets:
        raise ValueError(f"{subject}: P3a and CNV recordings do not share the same target sample coordinates.")

## P3a extraction

Epoch rejection uses the full EEG recording rather than the six-electrode ROI alone. All annotation-derived events enter epoching, and target epochs are selected afterward. Repeated events use MNE's `event_repeated="drop"` rule. The trial QC table records which targets survived and why other target epochs were rejected.

In [6]:
def extract_erp(raw, events, event_id, event_table, targets, subject):
    if missing := set(ERP_ROI) - set(raw.ch_names):
        raise ValueError(f"{subject}: missing P3a ROI channels {sorted(missing)}")
    metadata = event_table.merge(
        targets[["annotation_event_index", "trial"]], on="annotation_event_index", how="left", validate="one_to_one"
    )
    epochs = mne.Epochs(
        raw, events, event_id=event_id, tmin=ERP_TMIN, tmax=ERP_TMAX,
        baseline=BASELINE, reject=EEG_REJECT, reject_by_annotation=True,
        event_repeated="drop", metadata=metadata, preload=True, verbose="ERROR",
    )
    qc = targets.copy()
    qc["eeg_retained"] = qc["annotation_event_index"].isin(epochs.selection)
    qc["drop_reason"] = ["; ".join(epochs.drop_log[int(i)]) for i in qc["annotation_event_index"]]
    selected = epochs[epochs.metadata["event"].isin(EVENT_INFO.index).to_numpy()]
    window = (selected.times >= ERP_WINDOW[0]) & (selected.times <= ERP_WINDOW[1])
    if not window.any():
        raise ValueError("P3a measurement window has no samples.")
    rows = targets.set_index("annotation_event_index").loc[selected.selection].reset_index()
    rows["erp_amplitude_uv"] = selected.get_data(picks=ERP_ROI)[:, :, window].mean(axis=(1, 2)) * 1e6
    rows["sfreq_hz"] = float(raw.info["sfreq"])
    rows["first_samp"] = int(raw.first_samp)
    rows["window_first_s"] = selected.times[window][0]
    rows["window_last_s"] = selected.times[window][-1]
    rows["window_n_samples"] = int(window.sum())
    averages = []
    for condition in CONDITION_ORDER:
        choose = rows["condition"].eq(condition).to_numpy()
        if choose.any():
            average = selected[choose].average(picks="eeg")
            average.comment = condition
            averages.append(average)
    return rows, qc, averages

## Cue pairing and CNV extraction

Cue 12 indicates left and cue 13 indicates right. Each cue is matched to the first following target before another cue intervenes. The cue–target interval must be 1.25–1.60 s, inclusive. Cue direction and target direction are stored separately.

CNV epochs undergo EEG rejection followed by the saccade timing criterion. The minimum of 30 trials is checked separately in the 80% and 50% contexts after both exclusions. Within each context, left- and right-cue trials are pooled before averaging, giving equal weight to each retained trial.

In [7]:
def build_cue_pairs(event_table, targets, sfreq):
    """Pair each cue with the next target, stopping at the next cue."""
    if targets.duplicated(["target_sample", "event"]).any():
        raise ValueError("Target sample and event must uniquely identify a trial.")
    target_lookup = {
        (int(row.target_sample), int(row.event)): row._asdict()
        for row in targets.itertuples(index=False)
    }
    task = event_table.loc[
        event_table["event"].isin([*CUE_DIRECTIONS, *EVENT_INFO.index])
    ].sort_values("sample", kind="stable").to_dict("records")
    pairs, qc = [], []
    for position, cue in enumerate(task):
        cue_code = int(cue["event"])
        if cue_code not in CUE_DIRECTIONS:
            continue
        record = {
            "cue_annotation_event_index": int(cue["annotation_event_index"]),
            "cue_event": cue_code,
            "cue_sample": int(cue["sample"]),
            "actual_cue_direction": CUE_DIRECTIONS[cue_code],
            "pair_accepted": False,
            "pair_reason": "no_target_before_next_cue",
        }
        target_event = None
        for candidate in task[position + 1:]:
            if int(candidate["event"]) in CUE_DIRECTIONS:
                break
            if int(candidate["event"]) in EVENT_INFO.index:
                target_event = candidate
                break
        if target_event is None:
            qc.append(record)
            continue
        key = (int(target_event["sample"]), int(target_event["event"]))
        if key not in target_lookup:
            raise ValueError(f"Target {key} is absent from the aligned trial table.")
        target = target_lookup[key]
        record.update(target)
        record["soa_s"] = (key[0] - record["cue_sample"]) / sfreq
        if record["actual_cue_direction"] != int(target["cue_direction"]):
            raise ValueError(
                f"Cue direction disagrees with target event {key[1]} at sample {key[0]}."
            )
        if not SOA_RANGE[0] <= record["soa_s"] <= SOA_RANGE[1]:
            record["pair_reason"] = "soa_out_of_range"
        else:
            record.update(pair_accepted=True, pair_reason="accepted")
            pairs.append(record.copy())
        qc.append(record)
    return pd.DataFrame(pairs), pd.DataFrame(qc)


def extract_cnv(raw, event_table, targets, subject):
    """Extract trial CNV amplitudes and pooled probability-condition evokeds."""
    subject_qc = {"subject": subject, "included": False, "reason": ""}
    missing = sorted(set(CNV_REQUIRED_CHANNELS) - set(raw.ch_names))
    if missing:
        subject_qc.update(reason="missing_channels", missing_channels=",".join(missing))
        return pd.DataFrame(), subject_qc, pd.DataFrame(), []
    sfreq = float(raw.info["sfreq"])
    pairs, trial_qc = build_cue_pairs(event_table, targets, sfreq)
    trial_qc["subject"] = subject
    trial_qc["eeg_included"] = False
    trial_qc["eeg_drop_reason"] = "not_epoched"
    trial_qc["saccade_eligible"] = trial_qc.get(
        "cnv_saccade_ok", pd.Series(False, index=trial_qc.index)
    ).fillna(False).astype(bool)
    trial_qc["analysis_included"] = False
    subject_qc.update(n_cues=len(trial_qc), n_accepted_pairs=len(pairs))
    if pairs.empty:
        subject_qc["reason"] = "no_accepted_cue_target_pairs"
        return pd.DataFrame(), subject_qc, trial_qc, []

    # Retain cue side in epoch labels; pool both sides for CNV estimation.
    event_id = {
        "probability_80/cue_left": 1, "probability_80/cue_right": 2,
        "probability_50/cue_left": 3, "probability_50/cue_right": 4,
    }
    event_codes = np.where(pairs["probability"].eq(80), 1, 3)
    event_codes += pairs["actual_cue_direction"].eq(1).to_numpy(dtype=int)
    events = np.column_stack([
        pairs["cue_sample"].to_numpy(dtype=int), np.zeros(len(pairs), dtype=int), event_codes
    ])
    epochs = mne.Epochs(
        raw, events, event_id=event_id, tmin=CNV_TMIN, tmax=CNV_TMAX,
        baseline=BASELINE, picks="eeg", reject=EEG_REJECT,
        reject_by_annotation=True, metadata=pairs, preload=True,
        detrend=None, on_missing="ignore", verbose="ERROR",
    )
    epochs.drop_bad()
    qc_lookup = dict(zip(trial_qc["cue_annotation_event_index"], trial_qc.index))
    for position, reasons in enumerate(epochs.drop_log):
        index = qc_lookup[int(pairs.iloc[position]["cue_annotation_event_index"])]
        trial_qc.loc[index, "eeg_included"] = not bool(reasons)
        trial_qc.loc[index, "eeg_drop_reason"] = ";".join(reasons)
    subject_qc["n_eeg_clean"] = len(epochs)
    saccade_mask = epochs.metadata["cnv_saccade_ok"].to_numpy(dtype=bool)
    subject_qc["n_saccade_removed_after_eeg"] = int((~saccade_mask).sum())
    epochs = epochs[saccade_mask]
    counts = epochs.metadata["probability"].value_counts()
    for probability in (80, 50):
        subject_qc[f"n_probability_{probability}"] = int(counts.get(probability, 0))
    insufficient = [p for p in (80, 50) if counts.get(p, 0) < MIN_CNV_TRIALS]
    trial_qc["minimum_trials_met"] = not bool(insufficient)
    if insufficient:
        subject_qc["reason"] = "insufficient_trials_" + "_".join(map(str, insufficient))
        return pd.DataFrame(), subject_qc, trial_qc, []

    time_mask = (epochs.times >= CNV_WINDOW[0]) & (epochs.times <= CNV_WINDOW[1])
    if not time_mask.any():
        raise ValueError("No samples fall within the CNV measurement window.")
    trials = epochs.metadata.copy().reset_index(drop=True)
    trials["subject"] = subject
    trials["cnv_amplitude_uv"] = (
        epochs.get_data(picks=["Fz"])[:, 0, time_mask].mean(axis=1) * 1e6
    )
    trials["sfreq_hz"] = sfreq
    trials["first_samp"] = int(raw.first_samp)
    trials["window_first_s"] = epochs.times[time_mask][0]
    trials["window_last_s"] = epochs.times[time_mask][-1]
    trials["window_n_samples"] = int(time_mask.sum())
    retained_cues = set(trials["cue_annotation_event_index"])
    trial_qc["analysis_included"] = trial_qc["cue_annotation_event_index"].isin(retained_cues)
    subject_qc.update(included=True, reason="included", n_cnv_trials=len(trials))
    evokeds = []
    for probability in (80, 50):
        evoked = epochs[f"probability_{probability}"].average()
        evoked.comment = f"probability_{probability}"
        evokeds.append(evoked)
    return trials, subject_qc, trial_qc, evokeds

## Cohort processing and paired trials

The P3a cohort uses all configured participants. The three CNV exclusions are recorded separately, and further CNV exclusions are determined by channel availability, cue pairing, and retained-trial counts.

The paired table contains only trials with both a retained P3a response and an eligible CNV response. Standalone P3a and CNV tables retain their respective samples.

In [8]:
def prepare_eeg_features(reviewed, participant_ids):
    erp_parts, cnv_parts, erp_qc_parts, cnv_qc_parts = [], [], [], []
    input_rows, cnv_subject_rows, alignment_parts, evoked_files = [], [], [], {}
    for index, subject in enumerate(participant_ids, 1):
        print(f"{index}/{len(participant_ids)}: {subject}")
        erp_path = resolve_recording(ERP_FIF_DIR, ERP_FILE_PATTERN, subject)
        erp_raw = read_prepared_recording(erp_path)
        try:
            erp_events, erp_event_id, erp_event_table = recording_events(erp_raw)
            erp_targets = align_targets(erp_event_table, reviewed, subject)
            erp_rows, erp_qc, erp_averages = extract_erp(
                erp_raw, erp_events, erp_event_id, erp_event_table, erp_targets, subject
            )
            erp_parts.append(erp_rows)
            erp_qc_parts.append(erp_qc)
            alignment_parts.append(erp_targets.assign(sfreq_hz=erp_raw.info["sfreq"], first_samp=erp_raw.first_samp))
            input_rows.append(recording_metadata(erp_raw, erp_path, subject, "P3a"))
            if erp_averages:
                evoked_files[f"{subject}_erp-ave.fif"] = erp_averages
            cnv_path = resolve_recording(CNV_FIF_DIR, CNV_FILE_PATTERN, subject)
            cnv_raw = read_prepared_recording(cnv_path)
            try:
                _, _, cnv_event_table = recording_events(cnv_raw)
                cnv_targets = align_targets(cnv_event_table, reviewed, subject)
                verify_eeg_time_base(erp_raw, cnv_raw, erp_targets, cnv_targets, subject)
                input_rows.append(recording_metadata(cnv_raw, cnv_path, subject, "CNV"))
                cnv_rows, subject_qc, trial_qc, cnv_averages = extract_cnv(cnv_raw, cnv_event_table, cnv_targets, subject)
                cnv_subject_rows.append(subject_qc)
                if not trial_qc.empty: cnv_qc_parts.append(trial_qc)
                if subject_qc["included"]:
                    cnv_parts.append(cnv_rows)
                    evoked_files[f"{subject}_cnv-ave.fif"] = cnv_averages
            finally:
                cnv_raw.close()
        finally:
            erp_raw.close()

    erp_trials = pd.concat(erp_parts, ignore_index=True)
    if not cnv_parts:
        raise ValueError("No participants meet the CNV inclusion criteria; inspect inputs and CNV settings.")
    cnv_trials = pd.concat(cnv_parts, ignore_index=True)
    for label, table in (("P3a", erp_trials), ("CNV", cnv_trials)):
        if table.duplicated(["subject", "target_sample"]).any():
            raise ValueError(f"Duplicate participant/target samples in {label} table.")
    paired = erp_trials.merge(
        cnv_trials[["subject", "target_sample", "cnv_amplitude_uv", "cue_sample", "cue_event", "soa_s"]],
        on=["subject", "target_sample"], how="inner", validate="one_to_one",
    )
    erp_condition = erp_trials.groupby(["subject", "condition", "probability", "validity"], observed=True).agg(
        n_trials=("erp_amplitude_uv", "size"), mean_erp_amplitude_uv=("erp_amplitude_uv", "mean")
    ).reset_index()
    cnv_probability = cnv_trials.groupby(["subject", "probability"], observed=True).agg(
        n_trials=("cnv_amplitude_uv", "size"), mean_cnv_amplitude_uv=("cnv_amplitude_uv", "mean")
    ).reset_index()
    full_index = pd.MultiIndex.from_product([participant_ids, CONDITION_ORDER], names=["subject", "condition"])
    erp_counts = erp_trials.groupby(["subject", "condition"]).size().reindex(full_index, fill_value=0).rename("n_trials").reset_index()
    count_summary = pd.DataFrame([
        {"analysis": label, "participants": table["subject"].nunique(), "trials": len(table)}
        for label, table in (("P3a", erp_trials), ("CNV", cnv_trials), ("Paired P3a–CNV", paired))
    ])
    paired_counts = paired.groupby("subject").size().rename("n_paired").reset_index()
    tables = {
        "ERP_trials": erp_trials, "CNV_trials": cnv_trials, "ERP_CNV_pairs": paired,
        "ERP_condition": erp_condition, "CNV_probability": cnv_probability,
        "ERP_counts": erp_counts, "CNV_participant_QC": pd.DataFrame(cnv_subject_rows),
        "ERP_trial_QC": pd.concat(erp_qc_parts, ignore_index=True),
        "CNV_trial_QC": pd.concat(cnv_qc_parts, ignore_index=True) if cnv_qc_parts else pd.DataFrame(),
        "EEG_input_QC": pd.DataFrame(input_rows), "Event_alignment": pd.concat(alignment_parts, ignore_index=True),
        "Paired_counts": paired_counts, "Run_summary": count_summary,
        "Software_versions": software_versions, "Event_dictionary": EVENT_INFO.reset_index(),
    }
    return tables, evoked_files

In [9]:
eeg_tables = None
evoked_files = {}
if RUN_STUDY_ANALYSIS:
    participant_ids = [subject.strip().upper() for subject in PARTICIPANT_IDS]
    if len(participant_ids) != EXPECTED_N_PARTICIPANTS or len(set(participant_ids)) != len(participant_ids):
        raise ValueError("Cohort identifiers must be unique and match EXPECTED_N_PARTICIPANTS.")
    reviewed = load_saccade_summary(SACCADE_SUMMARY_FILE, participant_ids)
    eeg_tables, evoked_files = prepare_eeg_features(reviewed, participant_ids)
    eeg_tables["Saccade_input"] = pd.DataFrame([{
        "file": SACCADE_SUMMARY_FILE.name, "sha256": sha256_file(SACCADE_SUMMARY_FILE),
    }])
    settings = {
        "erp_epoch_s": (ERP_TMIN, ERP_TMAX), "cnv_epoch_s": (CNV_TMIN, CNV_TMAX),
        "baseline_s": BASELINE, "eeg_peak_to_peak_reject_v": EEG_REJECT["eeg"],
        "erp_roi": ERP_ROI, "erp_window_s": ERP_WINDOW, "cnv_channel": "Fz",
        "cnv_window_s": CNV_WINDOW, "cue_target_interval_s": SOA_RANGE,
        "minimum_cnv_trials_per_context": MIN_CNV_TRIALS,
        "cnv_saccade_rule": "detected; latency_ms >= 0; any direction",
        "erp_behavioral_exclusions": "none", "reject_by_annotation": True,
        "erp_event_repeated": "drop", "cnv_event_repeated": "error",
    }
    eeg_tables["Analysis_settings"] = pd.DataFrame([(k, repr(v)) for k, v in settings.items()], columns=["setting", "value"])
    display(eeg_tables["Run_summary"])
else:
    display(Markdown("Study extraction is disabled. Configure the inputs and set `RUN_STUDY_ANALYSIS = True`."))

Study extraction is disabled. Configure the inputs and set `RUN_STUDY_ANALYSIS = True`.

## Inspect counts and waveforms

Check the recorded filter settings and sample coordinates, the retained P3a counts in all four conditions, and the CNV exclusion reasons. The diagnostic waveforms show one participant's condition averages; they are not group results.

In [10]:
def plot_participant_waveforms(averages, subject):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    styles = {"valid_80": ("#202020", "-"), "invalid_80": ("#202020", "--"),
              "valid_50": ("#858585", "-"), "invalid_50": ("#858585", "--")}
    for evoked in averages.get(f"{subject}_erp-ave.fif", []):
        color, line = styles[evoked.comment]
        axes[0].plot(evoked.times * 1000, evoked.get_data(picks=ERP_ROI).mean(axis=0) * 1e6,
                     color=color, ls=line, label=evoked.comment.replace("_", " "))
    for evoked in averages.get(f"{subject}_cnv-ave.fif", []):
        probability = int(evoked.comment.rsplit("_", 1)[-1])
        axes[1].plot(evoked.times * 1000, evoked.get_data(picks=["Fz"])[0] * 1e6,
                     color="#202020" if probability == 80 else "#858585", label=f"{probability}%")
    for ax, window, title, reference in zip(axes, [ERP_WINDOW, CNV_WINDOW], ["P3a ROI", "CNV at Fz"], ["target", "cue"]):
        ax.axvspan(window[0] * 1000, window[1] * 1000, color="#EEEEEE", zorder=0)
        ax.axvline(0, color="#AAAAAA", lw=0.7)
        ax.axhline(0, color="#AAAAAA", lw=0.7)
        ax.set(title=title, xlabel=f"Time from {reference} onset (ms)", ylabel="Amplitude (µV)")
        if ax.get_legend_handles_labels()[0]: ax.legend(frameon=False, fontsize=8)
        else: ax.text(0.5, 0.5, "No retained average", ha="center", transform=ax.transAxes)
    fig.suptitle(f"Participant {subject}")
    return fig

if eeg_tables is not None:
    display(eeg_tables["EEG_input_QC"].drop(columns="sha256").head(10))
    display(eeg_tables["ERP_counts"].groupby("condition")["n_trials"].agg(["mean", "std", "min", "max"]).reindex(CONDITION_ORDER))
    display(eeg_tables["CNV_participant_QC"])
    display(eeg_tables["Paired_counts"].head(10))
    if SHOW_QC_PLOTS:
        diagnostic_figure = plot_participant_waveforms(evoked_files, QC_PARTICIPANT)
        plt.show()

## Export

`eeg_features.xlsx` contains the single-trial and participant-level tables, paired P3a–CNV trials, rejection records, settings, and input metadata. These are the inputs to the subsequent statistical notebooks.

With `SAVE_EVOKEDS = True`, the `evoked` subfolder also contains `<subject>_erp-ave.fif` with the available target conditions and `<subject>_cnv-ave.fif` with the two probability contexts for included CNV participants. Each average retains its channel geometry, time vector, and trial count.

Derived workbooks and FIF averages contain participant-level research data. Set a private output directory. Existing outputs are replaced only when `OVERWRITE_OUTPUTS = True`.

In [11]:
def export_eeg_features(tables, averages, output_dir, save_evokeds=True, overwrite=False):
    output_dir = Path(output_dir).expanduser()
    if "PATH_TO_" in str(output_dir):
        raise ValueError("Set OUTPUT_DIR before exporting.")
    if output_dir.resolve() in {ERP_FIF_DIR.resolve(), CNV_FIF_DIR.resolve()}:
        raise ValueError("Use an output directory separate from the EEG input directories.")
    workbook = output_dir / "eeg_features.xlsx"
    targets = [workbook] + ([output_dir / "evoked" / name for name in averages] if save_evokeds else [])
    existing = [path.name for path in targets if path.exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Outputs already exist: {existing[:10]}. Set OVERWRITE_OUTPUTS to replace them.")
    output_dir.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(workbook, engine="openpyxl") as writer:
        for sheet, table in tables.items():
            table.to_excel(writer, sheet_name=sheet, index=False)
    if save_evokeds:
        (output_dir / "evoked").mkdir(exist_ok=True)
        for name, evokeds in averages.items():
            mne.write_evokeds(output_dir / "evoked" / name, evokeds, overwrite=overwrite, verbose="ERROR")
    return pd.DataFrame({"file": [str(path.relative_to(output_dir)) for path in targets]})

if EXPORT_OUTPUTS:
    if eeg_tables is None:
        raise RuntimeError("Run EEG extraction before exporting.")
    display(export_eeg_features(eeg_tables, evoked_files, OUTPUT_DIR, SAVE_EVOKEDS, OVERWRITE_OUTPUTS))
else:
    print("Export disabled; no participant outputs written.")

Export disabled; no participant outputs written.


## Handoff

- `ERP_trials`: use `erp_amplitude_uv` for the target-locked mixed models.
- `CNV_trials`: use `cnv_amplitude_uv` for the cue-locked mixed models.
- `ERP_CNV_pairs`: use both amplitudes for the joint trial-level models.
- `ERP_condition`, `CNV_probability`, and condition averages: use for participant summaries and figures.

The cohort counts and trial counts must be checked against the manuscript after a study-data run. This notebook does not fit mixed models, standardize personality scores, or calculate significance tests.